In [1]:
# Librerias
# Manipulación de datos
import pandas as pd
import numpy as np

# Visualización
import plotly.express as px
import plotly.graph_objects as go

# Utilidades
from pathlib import Path

# Rutas
PATH_CUENTAS = Path(r"output\gold\master_cuentas_ahorro.parquet")
PATH_DPF = Path(r"output\gold\master_plazo_fijo.parquet")
PATH_TC = Path(r"output\gold\master_tarjetas_credito.parquet")

In [2]:
# Funciones: cargar dataset desde parquet
def cargar_parquet(path: Path) -> pd.DataFrame:
    """
    Carga un archivo parquet.

    Parameters
    ----------
    path : Path
        Ruta del archivo.

    Returns
    -------
    pd.DataFrame
        DataFrame cargado.
    """

    if not path.exists():
        raise FileNotFoundError(f"No existe el archivo: {path}")

    return pd.read_parquet(path)

def cargar_datasets():
    """
    Carga los datasets para cuentas, dpf y tarjetas

    Returns
    -------
    Lista de Dataframes
    """
    cuentas = cargar_parquet(PATH_CUENTAS)
    dpf = cargar_parquet(PATH_DPF)
    tarjetas = cargar_parquet(PATH_TC)

    return cuentas, dpf, tarjetas

def revisar_nulos(df, nombre):

    print(f"\n{nombre}")

    print(df.isnull().sum())

In [3]:
# Cargar DF
cuentas_df, dpf_df, tc_df = cargar_datasets()

print(f"Cuentas: {cuentas_df.shape}")

print(f"DPF: {dpf_df.shape}")

print(f"Tarjetas: {tc_df.shape}")

Cuentas: (17, 13)
DPF: (5, 9)
Tarjetas: (61, 32)


In [4]:
# Resumen de los datos
print("=" * 50)
print("RESUMEN DEL PROYECTO")
print("=" * 50)

print(f"Total de cuentas: {len(cuentas_df)}")

print(f"Total de depósitos: {len(dpf_df)}")

print(f"Total de tarjetas: {len(tc_df)}")

print(f"Bancos analizados: {cuentas_df['banco'].nunique()}")

print("=" * 50)

print("=" * 50)
print("NULOS")
print("=" * 50)
# Nulos
revisar_nulos(cuentas_df, "CUENTAS")

revisar_nulos(dpf_df, "DPF")

revisar_nulos(tc_df, "TARJETAS")
print("=" * 50)

# Estadísticos descriptivos
df_cuentas_describe = cuentas_df[
    [
        "trea_num",
        "monto_minimo_apertura",
        "mantenimiento_num"
    ]
].describe()

df_dpf_describe = dpf_df[
    [
        "trea_num",
        "monto_minimo_num"
    ]
].describe()


df_tc_describe = tc_df[
    [
        "ingreso_min_num",
        "membresia_num",
        "valor_neto_anual_recurrente_soles"
    ]
].describe()

print("=" * 50)
print("CUENTAS DE AHORRO - DESCRIPTIVO")
print("=" * 50)
print(df_cuentas_describe)

print("=" * 50)
print("DEPÓSITO A PLAZO FIJO - DESCRIPTIVO")
print("=" * 50)
print(df_dpf_describe)

print("=" * 50)
print("TARJETAS DE CRÉDITO - DESCRIPTIVO")
print("=" * 50)
print(df_tc_describe)

RESUMEN DEL PROYECTO
Total de cuentas: 17
Total de depósitos: 5
Total de tarjetas: 61
Bancos analizados: 5
NULOS

CUENTAS
banco                              0
producto_nombre                    0
trea_soles                         0
monto_minimo_apertura              0
mantenimiento_mensual              0
requisito_mantenimiento_gratis     0
retiros_gratuitos_cajero_propio    0
retiros_gratuitos_ventanilla       0
fecha_scraping                     0
url_origen                         0
mantenimiento_num                  0
trea_num                           0
es_costo_cero                      0
dtype: int64

DPF
banco                      0
producto                   0
moneda                     0
monto_minimo_apertura      0
trea_maxima_promocional    0
fecha_scraping             0
url_origen                 0
trea_num                   0
monto_minimo_num           0
dtype: int64

TARJETAS
nombre_tarjeta                        0
categoria                             0
membresia_anual

# GRÁFICOS QUE RESPONDEN PREGUNTAS DE NEGOCIO

## 📊 Gráfico 1: Comparativa de Tasas de Interés por Banco en Productos de Ahorro
### **¿Qué banco ofrecer la mejor tasa para productos de ahorro?​**


## 📌 Detalles del Análisis
* **Objetivo:** Obtener el producto de ahorro con mejor TREA. Sin distinguir entre depósito a plazo fijo y cuenta de ahorros.
* **Decisión de Diseño:** Tarjetas que contiene el producto con mejor TREA por separado y entre ambos productos, así como top 5 cuentas de ahorro y depósito a plazo fijo ordenados por TREA para que el usuario puede ver las diferentes opciones con las que cuenta en otros bancos. 
* **Librería Sugerida:** `Plotly Express`

In [ ]:
# 1. Unir DataFrames
todos_trea = pd.concat(
    [
        cuentas_df[["banco", "producto_nombre", "trea_num"]].rename(
            columns={"producto_nombre": "producto"}
        ),
        dpf_df[["banco", "producto", "trea_num"]],
    ],
    ignore_index=True,
)

# 2. Filtrar tasas mayores a 0
todos_trea = todos_trea[todos_trea["trea_num"] > 0].copy()

# 💡 SOLUCIÓN: Crear un nombre único combinando Producto + Banco para evitar superposición
todos_trea["producto_banco"] = (
    todos_trea["producto"] + "-" + todos_trea["banco"]
)

# 3. Ordenar de menor a mayor
todos_trea = todos_trea.sort_values(by="trea_num", ascending=False)

# 1. Extraer el producto Top 1
top_1 = todos_trea.iloc[0]

# 2. Crear la Card KPI con Plotly
fig = go.Figure()

fig.add_trace(
    go.Indicator(
        mode="number+delta",
        value=top_1["trea_num"],
        number={"suffix": "%", "font": {"size": 48, "color": "#10B981"}},
        title={
            "text": f"<b>🏆 {top_1['producto']}</b><br><span style='font-size:1.1em;color:#6B7280;'>🏦 {top_1['banco']}</span><br><br><span style='font-size:0.8em;color:#374151;'>Mayor TREA del Mercado</span>",
            "font": {"size": 18},
        },
    )
)

fig.update_layout(
    template="plotly_white",
    height=250,
    width=450,
    margin=dict(l=20, r=20, t=40, b=20),
)

fig.show()

In [12]:
# 1. Unir DataFrames
todos_trea = cuentas_df[["banco", "producto_nombre", "trea_num"]].rename(columns={"producto_nombre": "producto"})

# 2. Filtrar tasas mayores a 0
todos_trea = todos_trea[todos_trea["trea_num"] > 0].copy()

# 💡 SOLUCIÓN: Crear un nombre único combinando Producto + Banco para evitar superposición
todos_trea["producto_banco"] = (
    todos_trea["producto"] + "-" + todos_trea["banco"]
)

# 3. Ordenar de menor a mayor
todos_trea = todos_trea.sort_values(by="trea_num", ascending=False)

# 1. Extraer el producto Top 1
top_1 = todos_trea.iloc[0]

# 2. Crear la Card KPI con Plotly
fig = go.Figure()

fig.add_trace(
    go.Indicator(
        mode="number+delta",
        value=top_1["trea_num"],
        number={"suffix": "%", "font": {"size": 48, "color": "#10B981"}},
        title={
            "text": f"<b>🏆 {top_1['producto']}</b><br><span style='font-size:1.1em;color:#6B7280;'>🏦 {top_1['banco']}</span><br><br><span style='font-size:0.8em;color:#374151;'>Cuenta de Ahorro Mayor TREA del Mercado</span>",
            "font": {"size": 18},
        },
    )
)

fig.update_layout(
    template="plotly_white",
    height=250,
    width=450,
    margin=dict(l=20, r=20, t=40, b=20),
)

fig.show()

In [13]:
# 1. Unir DataFrames
todos_trea = dpf_df[["banco", "producto", "trea_num"]]

# 2. Filtrar tasas mayores a 0
todos_trea = todos_trea[todos_trea["trea_num"] > 0].copy()

# 💡 SOLUCIÓN: Crear un nombre único combinando Producto + Banco para evitar superposición
todos_trea["producto_banco"] = (
    todos_trea["producto"] + "-" + todos_trea["banco"]
)

# 3. Ordenar de menor a mayor
todos_trea = todos_trea.sort_values(by="trea_num", ascending=False)

# 1. Extraer el producto Top 1
top_1 = todos_trea.iloc[0]

# 2. Crear la Card KPI con Plotly
fig = go.Figure()

fig.add_trace(
    go.Indicator(
        mode="number+delta",
        value=top_1["trea_num"],
        number={"suffix": "%", "font": {"size": 48, "color": "#10B981"}},
        title={
            "text": f"<b>🏆 {top_1['producto']}</b><br><span style='font-size:1.1em;color:#6B7280;'>🏦 {top_1['banco']}</span><br><br><span style='font-size:0.8em;color:#374151;'>DPF con Mayor TREA del Mercado</span>",
            "font": {"size": 18},
        },
    )
)

fig.update_layout(
    template="plotly_white",
    height=250,
    width=450,
    margin=dict(l=20, r=20, t=40, b=20),
)

fig.show()

In [25]:
# 1. Unir DataFrames
todos_trea = pd.concat(
    [
        cuentas_df[["banco", "producto_nombre", "trea_num"]].rename(
            columns={"producto_nombre": "producto"}
        ),
        dpf_df[["banco", "producto", "trea_num"]],
    ],
    ignore_index=True,
)

# 2. Filtrar tasas mayores a 0
todos_trea = todos_trea[todos_trea["trea_num"] > 0].copy()

# 💡 SOLUCIÓN: Crear un nombre único combinando Producto + Banco para evitar superposición
todos_trea["producto_banco"] = (
    todos_trea["producto"] + "-" + todos_trea["banco"]
)

# 3. Ordenar de menor a mayor
todos_trea = todos_trea.sort_values(by="trea_num", ascending=False)

# 1. Filtrar los 5 primeros productos
top5_trea = (
    todos_trea.head(5).sort_values(by="trea_num", ascending=False).copy()
)
# 2. Calcular el valor máximo para fijar el límite superior del eje X en 100%
max_trea = top5_trea["trea_num"].max()

orden_eje_y = top5_trea["producto_banco"].tolist()[::-1]

# 3. Crear el gráfico de barras horizontales
fig = px.bar(
    top5_trea,
    x="trea_num",
    y="producto_banco",
    orientation="h",
    color="banco",
    text_auto=".2f",
    title="<b>Top 5 Productos Financieros con Mayor TREA</b>",
    labels={
        "trea_num": "TREA (%)",
        "producto_banco": "Producto - Banco",
        "banco": "Banco",
    },
)

# 4. Ajustar el diseño para simular barras gruesas y escala exacta
fig.update_traces(
    width=0.55,  # Grosor de las barras
    texttemplate="%{x:.2f}%",  # Muestra el porcentaje exacto al final de la barra
    textposition="outside",
)

fig.update_layout(
    template="plotly_white",
    xaxis=dict(
        range=[0, max_trea * 1.12],  # El máximo fija la escala y deja margen para el texto %
        title="TREA (%)",
    ),
    yaxis=dict(title="",categoryorder="array",categoryarray=orden_eje_y),  # Muestra el #1 arriba
    height=350,
    margin=dict(l=20, r=80, t=60, b=40),
)

fig.show()

## 📊 Gráfico 2: Comparativa de bancos con más cuentas de mantenimiento 0
### **¿Con qué banco tengo más opciones para abrir cuentas de ahorro de mantenimiento cero?​​**


## 📌 Detalles del Análisis
* **Objetivo:** Obtener visibilidad de la cantidad de cuentas de ahorro que tengan un mantenimiento 0, ordenado por banco. De esta forma el cliente puede saber en que institución tiene una mejor oferta de mantenimiento 0. Así mismo, los enlaces a estas cuentas.
* **Decisión de Diseño:** Gráfico de barras con la cantidad de cuentas de mantenimiento cero. Y, una tabla con las principales cuentas y sus enlaces. 
* **Librería Sugerida:** `Plotly Express`

In [40]:
# 1. Filtrar las cuentas con costo cero (mantenimiento_num == 0 o es_costo_cero == 'VERDADERO')
cuentas_costo_cero = cuentas_df[cuentas_df["mantenimiento_num"] == 0]

# 2. GroupBy por banco para contar la cantidad de cuentas y ordenar de MENOR a MAYOR
# (Para que Plotly muestre el banco con más cuentas en la parte superior)
df_ranking = (
    cuentas_costo_cero.groupby("banco", as_index=False)["producto_nombre"]
    .count()
    .rename(columns={"producto_nombre": "num_cuentas"})
    .sort_values(by="num_cuentas", ascending=True)
)

# 3. Crear el gráfico de barras horizontales
fig = px.bar(
    df_ranking,
    x="num_cuentas",
    y="banco",
    orientation="h",
    text="num_cuentas",
    title="🏛️ <b>Bancos con más cuentas de mantenimiento cero</b>",
    labels={"num_cuentas": "N° de cuentas", "banco": ""},
)

# 4. Estilizar el gráfico similar a tu imagen
fig.update_traces(
    marker_color="#0066CC",  # Azul corporativo del gráfico
    textposition="inside",  # Muestra el número dentro/al final de la barra
    width=0.6,
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="N° de cuentas",
    yaxis=dict(title=""),
    height=400,
    margin=dict(l=20, r=40, t=60, b=40),
)

fig.show()

## 📊 Gráfico 3: Costos de Membresía anual por Tarjeta
### **¿Qué banco tiene un mayor coste de membresía en sus tarjetas de crédito?​​**

## 📌 Detalles del Análisis
* **Objetivo:** Que el usuario pueda visualizar qué tarjetas tienen un mayor costo de membresía. Y pueda evitarlas, de ser el caso. Ignora membresías que son S/0.
* **Decisión de Diseño:** Gráfico de barras verticales que muestra las principales tarjetas con costo de membresía.
* **Librería Sugerida:** `Plotly Express`

In [59]:
# 1. Filtrar registros con membresía mayor a 0
df_membresia = tc_df[tc_df["membresia_num"] > 0][["nombre_tarjeta","banco","membresia_num"]].copy()

# 2. 💡 CREAR LA ETIQUETA ÚNICA (Tarjeta + Banco)
# Esto evita que Plotly agrupe o sobreponga tarjetas con nombres similares de distintos bancos
df_membresia["tarjeta_banco"] = (
    df_membresia["nombre_tarjeta"] + " - " + df_membresia["banco"]
)

# # 3. Ordenar de mayor a menor costo de membresía
df_membresia = df_membresia.sort_values(by="membresia_num", ascending=False)

df_membresia.head(40)

# # 4. Formato de texto con prefijo "S/ "
df_membresia["texto_membresia"] = df_membresia["membresia_num"].apply(
    lambda x: f"S/ {x:.0f}"
)

# 5. Lista con el orden explícito de las tarjetas
orden_tarjetas = df_membresia["tarjeta_banco"].tolist()

# 6. Crear el gráfico pasando 'tarjeta_banco' en el eje X
fig = px.bar(
    df_membresia,
    x="tarjeta_banco",  # 👈 Usar la columna combinada
    y="membresia_num",
    color="banco",
    text="texto_membresia",
    title="💳 <b>Costo de membresía anual por tarjeta</b>",
    labels={
        "tarjeta_banco": "Tarjeta",
        "membresia_num": "Membresía anual (S/)",
        "banco": "banco",
    },
)

# 7. Ajustar diseño y fijar orden
fig.update_traces(
    textposition="outside",
    width=0.6,
)

fig.update_layout(
    template="plotly_white",
    xaxis=dict(
        title="Tarjeta",
        tickangle=-35,
        categoryorder="array",  # 👈 Forzar el ordenamiento
        categoryarray=orden_tarjetas,  # 👈 Lista con el orden exacto de mayor a menor
    ),
    yaxis=dict(
        title="Membresía anual (S/)",
        range=[0, df_membresia["membresia_num"].max() * 1.15],
    ),
    height=480,
    margin=dict(l=20, r=20, t=60, b=120),
)

fig.show()

## 📊 Gráfico 4: Monto Mínimo de Apertura Requerido
### **¿Cuál es el producto financiero con menor barrera de entrada para ahorros de mediano/largo plazo y qué banco lo ofrece?​​**

## 📌 Detalles del Análisis
* **Objetivo:** Conocer el producto financiero que tiene una menor barrera de entrada pero para ahorros de mediano o largo plazo.
* **Decisión de Diseño:**  Gráfico de barras horizontales. Ordenado de menor a mayor. Así el usuario conoce cuáles son las instituciones financieras con menor monto mínimo de apertura.
* **Librería Sugerida:** `Plotly Express`

In [38]:
ranking_monto_minimo = (
    dpf_df[
        dpf_df["monto_minimo_num"] >= 0
    ][
        [
            "producto",
            "banco",
            "monto_minimo_num"
        ]
    ]
    .sort_values(
        by="monto_minimo_num",
        ascending=True
    )
)

ranking_monto_minimo["producto_banco"] = (
    ranking_monto_minimo["producto"]
    + " - "
    + ranking_monto_minimo["banco"]
)

fig = px.bar(
    ranking_monto_minimo,
    x="monto_minimo_num",
    y="producto_banco",
    orientation="h",
    text="monto_minimo_num",
    title="Monto mínimo de apertura por depósito a plazo fijo",
    labels={
        "monto_minimo_num": "Monto mínimo (S/)",
        "producto_banco": "Producto"
    }
)

fig.update_traces(
    texttemplate="S/ %{text:,.0f}",
    textposition="outside"
)

fig.update_layout(
    yaxis={"categoryorder": "total descending"},
    height=500
)

fig.show()

## 📊 Gráfico 5: 
### **¿Cuál es la mejor tarjeta "de entrada" para alguien con su primer sueldo, considerando que no tiene score ni ingresos altos?​**

## 📌 Detalles del Análisis
* **Objetivo:** Se busca entender cuáles son las tarjetas de crédito que tienen una menor barrera de entrada para aquellos usuarios que tienen sus primeros trabajos y ganan menos de S/3000.00, de esta forma puedan conocer la diversidad que tienene en los diferentes bancos.
* **Decisión de Diseño:** Gráfico de Dispersión (Scatter Plot) con técnica de dispersión aleatoria (jittering) en el eje X para que se pueda visualizar todas las tarjetas disponibles.
* **Librería Sugerida:** `Plotly Express`

In [61]:
# 1. Filtrar tarjetas con ingreso mínimo < 3000
df_entrada = tc_df[tc_df["ingreso_min_num"] < 3000].copy()

# 2. Generar el Jittering horizontal (pequeño ruido entre -2.5 y +2.5)
np.random.seed(42)  # Mantiene consistencia al ejecutar

df_entrada["membresia_num_jitter"] = df_entrada[
    "membresia_num"
] + np.random.uniform(-2.5, 2.5, size=len(df_entrada))
df_entrada["ingreso_min_num_jitter"] = df_entrada[
    "ingreso_min_num"
] + np.random.uniform(-2.5, 2.5, size=len(df_entrada))

# 3. Identificar la mejor tarjeta de entrada (menor ingreso, membresía = 0)
mejor_tarjeta = df_entrada.sort_values(
    by=["membresia_num", "ingreso_min_num"]
).iloc[0]

# 4. Crear el gráfico Scatter Plot
fig = px.scatter(
    df_entrada,
    x="ingreso_min_num_jitter",
    y="membresia_num_jitter",
    color="banco",
    hover_name="nombre_tarjeta",
    hover_data={
        "ingreso_min_num": ":.0f",
        "membresia_num": ":.0f",
        "ingreso_min_num_jitter": False,
    },
    text="nombre_tarjeta",
    title="<b>Tarjetas de entrada (ingreso mínimo < S/3,000): ingreso vs membresía</b>",
    labels={
        "ingreso_min_num_jitter": "Ingreso Mínimo Requerido (S/)",
        "membresia_num": "Membresía anual (S/)",
        "banco": "banco",
    },
)

# 5. Ajustar etiquetas de texto para evitar solapamiento pesado
fig.update_traces(
    textposition="top center",
    marker=dict(size=9),
)

fig.update_layout(
    template="plotly_white",
    xaxis=dict(
        title="Ingreso Mínimo Requerido (S/)",
        # Corregir el eje X para que muestre valores reales limpios en lugar del número con jitter
        tickvals=[1000, 1500, 2000, 2500, 3000],
        ticktext=["S/ 1,000", "S/ 1,500", "S/ 2,000", "S/ 2,500", "S/ 3,000"],
    ),
    yaxis=dict(title="Membresía anual (S/)"),
    height=500,
    margin=dict(l=20, r=20, t=60, b=40),
)

fig.show()

# 6. Imprimir la recomendación ejecutiva (Callout)
print(
    f"🏆 Mejor tarjeta de entrada: {mejor_tarjeta['nombre_tarjeta']} de {mejor_tarjeta['banco']} — "
    f"ingreso mínimo S/ {mejor_tarjeta['ingreso_min_num']:,.0f} y membresía S/ {mejor_tarjeta['membresia_num']:,.0f}."
)

🏆 Mejor tarjeta de entrada: Cero Membresia de nan — ingreso mínimo S/ 1,500 y membresía S/ 0.


# GRÁFICOS SECUNDARIOS

In [26]:
ranking_cuentas = (
    cuentas_df[
        cuentas_df["trea_num"] > 0
    ][
        [
            "producto_nombre",
            "banco",
            "trea_num"
        ]
    ]
    .sort_values(
        by="trea_num",
        ascending=True
    )
)

ranking_cuentas["producto_banco"] = (
    ranking_cuentas["producto_nombre"]
    + " - "
    + ranking_cuentas["banco"]
)

fig = px.bar(
    ranking_cuentas,
    x="trea_num",
    y="producto_banco",
    orientation="h",
    text="trea_num",
    title="Ranking de cuentas de ahorro según la TREA",
    labels={
        "trea_num": "TREA (%)",
        "producto_banco": "Producto"
    }
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside"
)

fig.update_layout(
    yaxis={
        "categoryorder": "total ascending"
    },
    height=600
)

fig.show()

In [27]:
ranking_dpf = (
    dpf_df[dpf_df["trea_num"] > 0][
        [
            "producto",
            "banco",
            "trea_num",
            "monto_minimo_num"
        ]
    ]
    .sort_values(
        by="trea_num",
        ascending=True
    )
)

ranking_dpf["producto_banco"] = (
    ranking_dpf["producto"]
    + " - "
    + ranking_dpf["banco"]
)

fig = px.bar(
    ranking_dpf,
    x="trea_num",
    y="producto_banco",
    orientation="h",
    text="trea_num",
    hover_data=["monto_minimo_num"],
    title="Ranking de depósitos a plazo según la TREA",
    labels={
        "trea_num": "TREA (%)",
        "producto_banco": "Producto"
    }
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside"
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    height=500
)

fig.show()

## 📊 Gráfico 2: Equilibrio entre Rentabilidad y Accesibilidad

### **¿Qué banco ofrece el mejor equilibrio entre rentabilidad y accesibilidad en los productos pasivos (cuentas de ahorro y depósitos a plazo)?**


## 📌 Detalles del Análisis

* **Objetivo:** Identificar bancos competitivos evaluando simultáneamente su rendimiento y estructura de costos.
* **Decisión de Diseño:** Gráfico de dispersión (*Scatter Plot*).
  * **Eje X:** Costo.
  * **Eje Y:** Rentabilidad.
  * **Interpretación:** Los mejores bancos aparecerán en la esquina superior izquierda (mayor rentabilidad y menor costo).
* **Librería Sugerida:** `Plotly Express`

In [28]:
# =========================
# RENTABILIDAD
# =========================

trea_cuentas = (
    cuentas_df
    .groupby("banco")["trea_num"]
    .mean()
    .reset_index(name="trea_promedio_cuentas")
)

trea_dpf = (
    dpf_df
    .groupby("banco")["trea_num"]
    .mean()
    .reset_index(name="trea_promedio_dpf")
)

rentabilidad = (
    trea_cuentas
    .merge(
        trea_dpf,
        on="banco",
        how="outer"
    )
)

rentabilidad["rentabilidad_promedio"] = (
    rentabilidad[
        [
            "trea_promedio_cuentas",
            "trea_promedio_dpf"
        ]
    ]
    .mean(axis=1)
)

# =========================
# ACCESIBILIDAD
# =========================

cuentas_tmp = cuentas_df.copy()

dpf_tmp = dpf_df.copy()

# Los DPF con TREA > 0 se consideran accesibles
dpf_tmp["es_costo_cero"] = dpf_tmp["trea_num"] > 0

productos = pd.concat(
    [
        cuentas_tmp[
            [
                "banco",
                "es_costo_cero"
            ]
        ],
        dpf_tmp[
            [
                "banco",
                "es_costo_cero"
            ]
        ]
    ],
    ignore_index=True
)

accesibilidad = (
    productos
    .groupby("banco")
    .agg(
        productos_gratuitos=(
            "es_costo_cero",
            "sum"
        ),
        total_productos=(
            "es_costo_cero",
            "count"
        )
    )
    .reset_index()
)

accesibilidad["accesibilidad"] = (
    accesibilidad["productos_gratuitos"]
    /
    accesibilidad["total_productos"]
)

# =========================
# TABLA FINAL
# =========================

indice_banco = (
    rentabilidad
    .merge(
        accesibilidad,
        on="banco"
    )
)

# Normalización

indice_banco["rentabilidad_norm"] = (
    (
        indice_banco["rentabilidad_promedio"]
        -
        indice_banco["rentabilidad_promedio"].min()
    )
    /
    (
        indice_banco["rentabilidad_promedio"].max()
        -
        indice_banco["rentabilidad_promedio"].min()
    )
)

indice_banco["accesibilidad_norm"] = (
    (
        indice_banco["accesibilidad"]
        -
        indice_banco["accesibilidad"].min()
    )
    /
    (
        indice_banco["accesibilidad"].max()
        -
        indice_banco["accesibilidad"].min()
    )
)

indice_banco["indice_banco"] = (
    0.5
    *
    indice_banco["rentabilidad_norm"]
    +
    0.5
    *
    indice_banco["accesibilidad_norm"]
)

indice_banco = indice_banco.sort_values(
    "indice_banco",
    ascending=False
)

indice_banco

,banco,trea_promedio_cuentas,trea_promedio_dpf,rentabilidad_promedio,productos_gratuitos,total_productos,accesibilidad,rentabilidad_norm,accesibilidad_norm,indice_banco
4,Scotiabank,7.150000,4.50,5.825000,3,3,1.000000,1.000000,1.00,1.000000
3,Interbank,0.000000,4.50,2.250000,4,4,1.000000,0.243386,1.00,0.621693
1,BCP,4.883333,2.77,3.826667,3,4,0.750000,0.577072,0.25,0.413536
0,BBVA,0.187500,4.40,2.293750,4,5,0.800000,0.252646,0.40,0.326323
2,BanBif,2.200000,0.00,1.100000,4,6,0.666667,0.000000,0.00,0.000000


In [29]:
fig = px.scatter(
    indice_banco,
    x="accesibilidad_norm",
    y="rentabilidad_norm",
    size="indice_banco",
    text="banco",
    hover_data=[
        "rentabilidad_promedio",
        "accesibilidad",
        "indice_banco"
    ],
    title="Rentabilidad vs accesibilidad por banco",
    labels={
        "accesibilidad_norm": "Accesibilidad",
        "rentabilidad_norm": "Rentabilidad"
    }
)

fig.update_traces(
    textposition="top center"
)

fig.update_layout(
    plot_bgcolor="white",
    height=600
)

fig.show()

## 📊 Gráfico 3: Tarjetas de Crédito con Mayor Valor Anual

### **¿Qué tarjeta de crédito ofrece los mayores beneficios económicos en el largo plazo?**


## 📌 Detalles del Análisis

* **Objetivo:** Identificar las tarjetas que generan un mayor retorno económico para el usuario.
* **Indicador Utilizado:** `valor_neto_anual_recurrente`
* **Decisión de Diseño:** Gráfico de barras horizontales.
  * **Eje X:** Valor neto anual.
  * **Eje Y:** Tarjeta y Banco.
  * **Interpretación:** Las tarjetas ubicadas en la parte superior del ranking ofrecen un mayor beneficio económico anual.
* **Librería Sugerida:** `Plotly Express`

In [30]:
ranking_tc = tc_df.copy()

ranking_tc["valor_neto_anual_recurrente_soles"] = pd.to_numeric(
    ranking_tc["valor_neto_anual_recurrente_soles"],
    errors="coerce"
)

ranking_tc = ranking_tc.dropna(
    subset=["valor_neto_anual_recurrente_soles"]
)

ranking_tc = ranking_tc[
    (ranking_tc["valor_neto_anual_recurrente_soles"] > 0) &
    (ranking_tc["banco"] != "None") &
    (ranking_tc["banco"].notna()) &
    (ranking_tc["nombre_tarjeta"] != "None")
]

ranking_tc["tarjeta_banco"] = (
    ranking_tc["nombre_tarjeta"].astype(str)
    + " - "
    + ranking_tc["banco"].astype(str)
)

ranking_tc = ranking_tc.sort_values(
    by="valor_neto_anual_recurrente_soles",
    ascending=True
)

fig = px.bar(
    ranking_tc,
    x="valor_neto_anual_recurrente_soles",
    y="tarjeta_banco",
    orientation="h",
    title="Tarjetas con mayor valor neto anual",
    labels={
        "valor_neto_anual_recurrente_soles": "Valor neto anual",
        "tarjeta_banco": "Tarjeta"
    }
)

fig.update_traces(
    text=ranking_tc["valor_neto_anual_recurrente_soles"].round(0),
    textposition="outside"
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    height=700
)

fig.show()

## 📊 Gráfico 5: Ranking de Costo de Mantenimiento

### **¿Qué cuentas de ahorro cobran comisión de mantenimiento y cuánto?**

## 📌 Detalles del Análisis

* **Objetivo:** Comparar el monto del costo de mantenimiento entre las cuentas que sí lo cobran, para identificar cuáles son las más costosas.
* **Decisión de Diseño:** Gráfico de barras horizontales (*Bar Chart*).
* **Librería Sugerida:** `Plotly Express`

In [32]:
ranking_mantenimiento = (
    cuentas_df[
        cuentas_df["mantenimiento_num"] > 0
    ][
        [
            "producto_nombre",
            "banco",
            "mantenimiento_num"
        ]
    ]
    .sort_values(
        by="mantenimiento_num",
        ascending=True
    )
)

ranking_mantenimiento["producto_banco"] = (
    ranking_mantenimiento["producto_nombre"]
    + " - "
    + ranking_mantenimiento["banco"]
)

fig = px.bar(
    ranking_mantenimiento,
    x="mantenimiento_num",
    y="producto_banco",
    orientation="h",
    text="mantenimiento_num",
    title="Cuentas con costo de mantenimiento mayor a 0",
    labels={
        "mantenimiento_num": "Costo de mantenimiento (S/)",
        "producto_banco": "Producto"
    }
)

fig.update_traces(
    texttemplate="S/ %{text:.2f}",
    textposition="outside"
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    height=500
)

fig.show()

## 📊 Gráfico 6: Cuentas con y sin Costo de Mantenimiento

### **¿Qué proporción de cuentas de ahorro tiene costo de mantenimiento?**

## 📌 Detalles del Análisis

* **Objetivo:** Cuantificar y comparar la cantidad de cuentas de ahorro gratuitas frente a las que aplican costo de mantenimiento.
* **Decisión de Diseño:** Gráfico de barras horizontales (*Bar Chart*).
* **Librería Sugerida:** `Plotly Express`

In [33]:
# 1. Lógica de costo idéntica a Streamlit
if "es_costo_cero" in cuentas_df.columns:
    cuentas_df["tiene_costo"] = ~cuentas_df["es_costo_cero"].astype(str).str.upper().isin(["TRUE", "VERDADERO"])
elif "mantenimiento_num" in cuentas_df.columns:
    cuentas_df["tiene_costo"] = cuentas_df["mantenimiento_num"] > 0
else:
    cuentas_df["tiene_costo"] = False

# 2. Agrupación y cálculo de cantidades
resumen_costo = (
    cuentas_df.groupby("tiene_costo")
    .size()
    .reset_index(name="cantidad")
)

# 3. Mapeo de categorías exactamente como en Streamlit
resumen_costo["categoria"] = resumen_costo["tiene_costo"].map(
    {True: "Con Costo", False: "Sin Costo (Gratis)"}
)

# 4. Cálculo de porcentaje
total = resumen_costo["cantidad"].sum()
resumen_costo["porcentaje"] = (resumen_costo["cantidad"] / total) * 100 if total > 0 else 0

# 5. Gráfico de Plotly
fig = px.bar(
    resumen_costo,
    x="cantidad",
    y="categoria",
    orientation="h",
    text=resumen_costo["porcentaje"].round(1).astype(str) + "%",
    title="Cuentas con y sin costo de mantenimiento",
    labels={
        "cantidad": "Cantidad de cuentas",
        "categoria": ""
    },
    color="categoria",
    color_discrete_map={
        "Sin Costo (Gratis)": "#2ca02c",
        "Con Costo": "#d62728"
    }
)

fig.update_traces(
    textposition="outside"
)

fig.update_layout(
    height=350,
    showlegend=False
)

fig.show()

## 📊 Gráfico 8: TREA vs Monto Mínimo Requerido

### **¿Existe relación entre la rentabilidad y el monto mínimo exigido?**

## 📌 Detalles del Análisis

* **Objetivo:** Explorar si los depósitos a plazo fijo con mayor TREA exigen también un monto mínimo de apertura más alto, para evaluar la relación costo-beneficio de cada producto.
* **Decisión de Diseño:** Gráfico de dispersión (*Scatter Plot*).
* **Librería Sugerida:** `Plotly Express`

In [32]:
fig = px.scatter(
    dpf_df[dpf_df["trea_num"] > 0],
    x="monto_minimo_num",
    y="trea_num",
    color="banco",
    text="producto",
    hover_data=["banco", "producto"],
    title="TREA vs Monto mínimo requerido en depósitos a plazo fijo",
    labels={
        "monto_minimo_num": "Monto mínimo (S/)",
        "trea_num": "TREA (%)"
    }
)

fig.update_traces(
    textposition="top center"
)

fig.update_layout(
    plot_bgcolor="white",
    height=600
)

fig.show()

## 📊 Gráfico 9: Tarjeta con Mejor Valor Neto Anual

### **¿Qué tarjeta de crédito ofrece el mayor valor neto para el usuario?**

## 📌 Detalles del Análisis

* **Objetivo:** Identificar de forma inmediata la tarjeta de crédito con el mayor valor neto anual recurrente, considerando membresía, millas y beneficios.
* **Decisión de Diseño:** Indicador numérico destacado (*KPI Card*).
* **Librería Sugerida:** `Plotly Graph Objects (Indicator)`

In [33]:
tc_df_valido = tc_df.copy()

tc_df_valido["valor_neto_anual_recurrente_soles"] = pd.to_numeric(
    tc_df_valido["valor_neto_anual_recurrente_soles"],
    errors="coerce"
)

tc_df_valido = tc_df_valido.dropna(
    subset=["valor_neto_anual_recurrente_soles"]
)

mejor_tc = tc_df_valido.loc[
    tc_df_valido["valor_neto_anual_recurrente_soles"].idxmax()
]

fig = go.Figure(
    go.Indicator(
        mode="number",
        value=mejor_tc["valor_neto_anual_recurrente_soles"],
        number={"prefix": "S/ ", "valueformat": ",.0f"},
        title={
            "text": f"<b>{mejor_tc['nombre_tarjeta']}</b><br><span style='font-size:0.7em'>{mejor_tc['banco']}</span><br><span style='font-size:0.6em'>Mejor Valor Neto Anual</span>"
        }
    )
)

fig.update_layout(height=300)

fig.show()

## 📊 Gráfico 10: Valor Neto Anual por Tarjeta según Segmento de Gasto

### **¿Qué tarjeta conviene más y a qué segmento de gasto pertenece?**

## 📌 Detalles del Análisis

* **Objetivo:** Comparar el valor neto anual de todas las tarjetas de crédito, identificando visualmente a qué segmento de gasto (Básico, Medio, Premium) pertenece cada una.
* **Decisión de Diseño:** Gráfico de barras horizontales (*Bar Chart*), coloreado por segmento.
* **Librería Sugerida:** `Plotly Express`

In [36]:
ranking_tc_segmento = tc_df.copy()

ranking_tc_segmento["valor_neto_anual_recurrente_soles"] = pd.to_numeric(
    ranking_tc_segmento["valor_neto_anual_recurrente_soles"],
    errors="coerce"
)

ranking_tc_segmento = ranking_tc_segmento.dropna(
    subset=["valor_neto_anual_recurrente_soles"]
)

ranking_tc_segmento = ranking_tc_segmento[
    (ranking_tc_segmento["valor_neto_anual_recurrente_soles"] > 0) &
    (ranking_tc_segmento["nombre_tarjeta"] != "None") &
    (ranking_tc["banco"].notna()) &
    (ranking_tc_segmento["nombre_tarjeta"].notna())
]

ranking_tc_segmento["tarjeta_banco"] = (
    ranking_tc_segmento["nombre_tarjeta"].astype(str)
    + " - "
    + ranking_tc_segmento["banco"].astype(str)
)

ranking_tc_segmento = ranking_tc_segmento.sort_values(
    by="valor_neto_anual_recurrente_soles",
    ascending=True
)

fig = px.bar(
    ranking_tc_segmento,
    x="valor_neto_anual_recurrente_soles",
    y="tarjeta_banco",
    orientation="h",
    color="segmento_gasto",
    text="valor_neto_anual_recurrente_soles",
    title="Valor neto anual por tarjeta según segmento de gasto",
    labels={
        "valor_neto_anual_recurrente_soles": "Valor neto anual (S/)",
        "tarjeta_banco": "Tarjeta",
        "segmento_gasto": "Segmento"
    }
)

fig.update_traces(
    texttemplate="S/ %{text:,.0f}",
    textposition="outside"
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    height=700
)

fig.show()

## 📊 Gráfico 11: Tabla de Beneficios Principales

### **¿Qué beneficios ofrece cada tarjeta de crédito?**

## 📌 Detalles del Análisis

* **Objetivo:** Presentar de forma clara y comparable los beneficios clave y el beneficio principal de cada tarjeta de crédito, junto con su banco y categoría.
* **Decisión de Diseño:** Tabla interactiva (*Table*).
* **Librería Sugerida:** `Plotly Graph Objects (Table)`

In [38]:
tabla_beneficios = tc_df[
    [
        "nombre_tarjeta",
        "banco",
        "categoria",
        "beneficio_principal",
        "beneficios_clave"
    ]
].dropna(subset=["nombre_tarjeta", "banco"])

fig = go.Figure(
    data=[
        go.Table(
            header=dict(
                values=["Tarjeta", "Banco", "Categoría", "Beneficio Principal", "Beneficios Clave"],
                fill_color="#2c3e50",
                font=dict(color="white", size=12),
                align="left"
            ),
            cells=dict(
                values=[
                    tabla_beneficios["nombre_tarjeta"],
                    tabla_beneficios["banco"],
                    tabla_beneficios["categoria"],
                    tabla_beneficios["beneficio_principal"],
                    tabla_beneficios["beneficios_clave"]
                ],
                fill_color="#f8f9fa",
                align="left"
            )
        )
    ]
)

fig.update_layout(
    title="Beneficios principales por tarjeta de crédito",
    height=500
)

fig.show()